## Part 1 — Titanic: Binary Classification (EDA)

In [ ]:

import os, sys, warnings, json, math, itertools, textwrap
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

DATA_TITANIC = "/kaggle/input/titanic"
train_path = os.path.join(DATA_TITANIC, "train.csv")
test_path  = os.path.join(DATA_TITANIC, "test.csv")

df = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(df.shape, df_test.shape)
df.head()


In [ ]:
# thông tin và missing values
display(df.info())
print("\nMissing values per column:\n", df.isna().sum())


In [ ]:

# tóm tắt số liệu
display(df.describe(include=[np.number]))

# phân phối
counts = df['Survived'].value_counts().sort_index()
plt.figure()
counts.plot(kind='bar')
plt.title("Target distribution: Survived (0/1)")
plt.xlabel("Survived")
plt.ylabel("Count")
plt.show()


In [ ]:

# mooiso quân hệ giữa survival với sex / pclass
plt.figure()
df.groupby('Sex')['Survived'].mean().plot(kind='bar')
plt.title("Mean Survival Rate by Sex")
plt.xlabel("Sex"); plt.ylabel("Mean Survived")
plt.show()

plt.figure()
df.groupby('Pclass')['Survived'].mean().plot(kind='bar')
plt.title("Mean Survival Rate by Pclass")
plt.xlabel("Pclass"); plt.ylabel("Mean Survived")
plt.show()


### Preprocessing & Model

In [ ]:

# chọn features
target = 'Survived'
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X = df[features].copy()
y = df[target].copy()

# split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_features = ['Age', 'SibSp', 'Parch', 'Fare']
cat_features = ['Pclass', 'Sex', 'Embarked']

# preprocess: impute + scale numeric; impute + onehot categorical
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ]
)

# so sánh 2 model
logreg_clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=200))
])

rf_clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=42))
])

models = {
    "LogReg": logreg_clf,
    "RandomForest": rf_clf
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_valid)
    acc = accuracy_score(y_valid, preds)
    prec = precision_score(y_valid, preds, zero_division=0)
    rec = recall_score(y_valid, preds, zero_division=0)
    cm = confusion_matrix(y_valid, preds)
    print(f"\n=== {name} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print("Confusion Matrix:\n", cm)
    print("\nClassification Report:\n", classification_report(y_valid, preds, zero_division=0))
    results[name] = {"acc":acc, "prec":prec, "rec":rec, "cm":cm.tolist()}

pd.DataFrame(results).T



**Reflection (Titanic)**

-Các đặc trưng như **Sex** và **Pclass** thường có ảnh hưởng mạnh đến khả năng sống sót.

-Em đã xử lý các giá trị bị thiếu (dùng trung vị cho dữ liệu số và giá trị xuất hiện nhiều nhất cho dữ liệu phân loại), mã hóa one-hot cho các biến phân loại, và chuẩn hóa các biến số.

-Logistic Regression mang lại khả năng giải thích rõ ràng, trong khi Random Forest có thể nắm bắt các mối quan hệ phi tuyến tính.

-Các hướng cải thiện: hyperparameter tuning, feature engineering (ví dụ: trích xuất chức danh từ tên, quy mô gia đình), và kết hợp mô hình (ensembling) có thể giúp tăng độ chính xác.


## Part 2 — HAM10000: Imbalanced Data Handling

In [ ]:

import os, glob, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

DATA_HAM = "/kaggle/input/skin-cancer-mnist-ham10000"
meta_path = os.path.join(DATA_HAM, "HAM10000_metadata.csv")

meta = pd.read_csv(meta_path)
print(meta.shape)
meta.head()


In [ ]:

# class distribution
cls_counts = meta['dx'].value_counts().sort_values(ascending=False)
plt.figure()
cls_counts.plot(kind='bar')
plt.title("HAM10000 — Class Distribution (dx)")
plt.xlabel("Class (dx)")
plt.ylabel("Count")
plt.show()

img_dirs = [os.path.join(DATA_HAM, "HAM10000_images_part_1"),
            os.path.join(DATA_HAM, "HAM10000_images_part_2")]

def find_img_path(image_id):
    filename_jpg = image_id + ".jpg"
    for d in img_dirs:
        p = os.path.join(d, filename_jpg)
        if os.path.exists(p):
            return p
    return None

meta['image_path'] = meta['image_id'].apply(find_img_path)
print("Missing paths:", meta['image_path'].isna().sum())
meta = meta.dropna(subset=['image_path']).reset_index(drop=True)
meta.head()


In [ ]:

# xuất 1 ít mẫu
import random
classes = meta['dx'].unique().tolist()
n_per_class = 3
plt.figure(figsize=(12, 4 * len(classes)))
idx = 1
for c in classes:
    sub = meta[meta['dx'] == c].sample(min(n_per_class, len(meta[meta['dx']==c])), random_state=42)
    for _, row in sub.iterrows():
        img = plt.imread(row['image_path'])
        plt.subplot(len(classes), n_per_class, idx)
        plt.imshow(img)
        plt.title(f"{c}")
        plt.axis('off')
        idx += 1
plt.tight_layout()
plt.show()


### Train/Valid split & visualize imbalance

In [ ]:

# stratified split
train_df, val_df = train_test_split(
    meta[['image_path','dx']],
    test_size=0.2,
    random_state=42,
    stratify=meta['dx']
)

def plot_dist(df, title):
    plt.figure()
    df['dx'].value_counts().sort_values(ascending=False).plot(kind='bar')
    plt.title(title)
    plt.xlabel("Class (dx)")
    plt.ylabel("Count")
    plt.show()

plot_dist(train_df, "Train distribution (gốc)")
plot_dist(val_df,   "Validation distribution")


### Rebalancing (Oversampling) & Augmentation

In [ ]:

# oversampling
from sklearn.utils import resample

max_count = train_df['dx'].value_counts().max()
cap = min(max_count, 2000) 

balanced_parts = []
for c, grp in train_df.groupby('dx'):
    if len(grp) < cap:
        up = resample(grp, replace=True, n_samples=cap, random_state=42)
        balanced_parts.append(up)
    else:
        balanced_parts.append(grp.sample(cap, random_state=42))
train_bal = pd.concat(balanced_parts).sample(frac=1.0, random_state=42).reset_index(drop=True)

plot_dist(train_bal, "Train distribution AFTER oversampling")
print("Train size gốc:", len(train_df), " → Balanced train size:", len(train_bal))
train_bal.head()


### Small CNN trên balanced data

In [ ]:

import tensorflow as tf
print("TF version:", tf.__version__)

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# label mapping
classes_sorted = sorted(train_bal['dx'].unique().tolist())
cls2idx = {c:i for i,c in enumerate(classes_sorted)}
idx2cls = {i:c for c,i in cls2idx.items()}

def df_to_dataset(df, shuffle=True):
    paths = df['image_path'].values
    labels = df['dx'].map(cls2idx).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(path, y):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        return img, tf.cast(y, tf.int32)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(2048, seed=42)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

# augmentation
data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip(),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

def df_to_augmented_dataset(df, shuffle=True):
    base = df_to_dataset(df, shuffle=shuffle)
    return base.map(lambda x, y: (data_augment(x, training=True), y), num_parallel_calls=AUTOTUNE)

train_ds = df_to_augmented_dataset(train_bal, shuffle=True)
val_ds   = df_to_dataset(val_df, shuffle=False)

# CNN
inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = tf.keras.layers.Conv2D(32, 3, activation='relu')(inputs)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(64, 3, activation='relu')(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(128, 3, activation='relu')(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(classes_sorted), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


In [ ]:

# train 
EPOCHS = 2
hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)


In [ ]:

# đánh giá và xuất kết quả
y_true = []
y_pred = []
for xb, yb in val_ds:
    probs = model.predict(xb, verbose=0)
    preds = probs.argmax(axis=1)
    y_true.extend(yb.numpy().tolist())
    y_pred.extend(preds.tolist())

print("Classification Report (Validation):")
print(classification_report(y_true, y_pred, target_names=[idx2cls[i] for i in range(len(classes_sorted))]))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes_sorted))))
plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix (Validation)")
plt.colorbar()
tick_marks = np.arange(len(classes_sorted))
plt.xticks(tick_marks, [idx2cls[i] for i in range(len(classes_sorted))], rotation=45, ha='right')
plt.yticks(tick_marks, [idx2cls[i] for i in range(len(classes_sorted))])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()



**Reflection (HAM10000)**

- Tập data bị mất cân bằng nghiêm trọng giữa các nhãn chẩn đoán (dx).

- Em đã trực quan hóa class distribution, sau đó cân bằng lại tập train bằng cách oversampling, và áp dụng data augmentation để cải thiện khả năng tổng quát hóa của model.

- CNN được huấn luyện trong thời gian ngắn nhằm minh họa quy trình train end-to-end.

- Các phương án thay thế: loss function có trọng số theo lớp (class-weighted loss), focal loss, hoặc lấy mẫu có trọng số (weighted sampling) trong pipeline dữ liệu.
